In [1]:
import sys
print(sys.path) 

sys.path.insert(0, "/mnt/nas/pritish/CMUVERA_IR_ref")
sys.path.insert(0, "/raid/infolab/pritish/CMUVERA_IR_ref")
print(sys.path)

['', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/setuptools/_vendor']
['/raid/infolab/pritish/CMUVERA_IR_ref', '/mnt/nas/pritish/CMUVERA_IR_ref', '', '/mnt/home/pritish/.local/lib/python3.11/site-packages', '/mnt/nas/pritish', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python311.zip', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/lib-dynload', '/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages', '/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT', '/mnt/nas/pritish/conda/.c

In [2]:
import torch
import os
import numpy as np
from omegaconf import OmegaConf
import time
import logging
from src.utils import set_seed,set_seed_from_checkpoint, load, save
from tqdm import tqdm
import random
from numpy.random import default_rng
import submodlib
import pickle

from src.dataloader import get_dataloader
from src.embedder import ColBERTEmbedder

from src.utils import partial_chamfer_sim_batched_with_rerank
import torch.multiprocessing as mp

# 1) Make sure the env var is set *inside* Python too
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# 2) Turn on PyTorch’s deterministic‐only mode
torch.use_deterministic_algorithms(True)

from loguru import logger

/mnt/nas/pritish/conda/.conda/envs/knmuvera/lib/python3.11/site-packages/beir/util.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


In [3]:
%load_ext autoreload
%autoreload 1

%aimport src.endtoend
%aimport make_plots

In [4]:
config = {
    "k": 15,
    "method": "v0",
    "bucket_size": 0,
    "dataset_name": "nfcorpus",
    "mv_type": 'colbertv2-plaid'
}

In [5]:
def make_config(config):
    sys.argv = ["", f"k={config['k']}", f"method={config['method']}",
                f"baseline.bucket_size={config['bucket_size']}",
                f"data.dataset_name={config['dataset_name']}",
                f"embedder.mv_type={config['mv_type']}",
                f"embedder.mode={config.get('mode', 'mem')}"
    ]
    print(sys.argv)
    
    cli_conf = OmegaConf.from_cli()
    file_config = OmegaConf.load("configs/config.yaml")

    conf = OmegaConf.merge(file_config, cli_conf)

    return conf

In [6]:
os.chdir("CMUVERA_IR_ref")

In [7]:
conf = make_config(config)

['', 'k=15', 'method=v0', 'baseline.bucket_size=0', 'data.dataset_name=nfcorpus', 'embedder.mv_type=colbertv2-plaid', 'embedder.mode=mem']


In [19]:
conf

{'method': 'v0', 'k': 15, 'num_rh': 8, 'seed': 0, 'query_batch_size': 512, 'corpus_batch_size': 5120, 'dtype': 'float32', 'data': {'loader_type': 'beir', 'dataset_name': 'nfcorpus'}, 'baseline': {'bucket_size': 0, 'seed': 17, 'use_aug': False}, 'device': 'cuda:0', 'posting': {'variety': 'mp', 'aggregation': 'threshold', 'update_list': True, 'update_retr_list': True, 'numpy': False, 'threshold_values': [0.1, 0.3, 0.5, 0.7, 0.9], 'device': 'cuda:0', 'lsh': {'id': 0, 'type': 'base', 'input_dim': -1, 'hash_dim': 12, 'create_tables': False, 'seed': 22, 'format': 'int'}}, 'retriever': {'type': 'bf', 'muvera_bucket_size': 100, 'mode': 'disk', 'corpus_batch_size': 10240, 'num_batches': -1, 'dtype': 'float32'}, 'embedder': {'recompute_emb': False, 'pretrained': True, 'model_name': 'bert-base-uncased', 'save_embeddings': True, 'batch_size': 1024, 'mode': 'mem', 'type': 'BERT', 'mv_type': 'colbertv2-plaid', 'emb_dim': 128, 'query_batch_size': None}, 'muvera': {'type': 'base', 'in_emb_dim': -1, 'c

In [8]:
def get_method(config):
    name = config.method
    if name == "v0":
        return src.endtoend.GreedyBaseline_v0(config)
    elif name=="sml":
        return src.endtoend.GreedyBaseline_submodlib(config)
    else:
        raise ValueError(f"Unknown method: {name}")

In [9]:
retriever = get_method(conf)
retriever.run()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3633/3633 [00:00<00:00, 114439.08it/s]
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/mnt/nas/pritish/CMUVERA_IR_ref/ColBERT/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: Do Cholesterol Statin Drugs Cause Breast Cancer?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  2079, 16480,  4244, 27833, 28093,  2378,  5850,  3426,
         7388,  4456,  1029,   102,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/nfcorpus/BERT/corpus/compressed_128/batch_0.0.pkl
0 1 loaded


/mnt/nas/pritish/CMUVERA_IR_ref/src/embedder.py:327: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cemb = torch.load(self.embedding_path(f"compressed_{self.config.emb_dim}",

0 2 loaded


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 323/323 [01:04<00:00,  5.00it/s]


[[(1, 22.11961555480957),
  (3097, 26.389442443847656),
  (2126, 27.00503158569336),
  (1373, 27.201814651489258),
  (2315, 27.21265983581543),
  (3057, 27.217079162597656),
  (0, 27.221275329589844),
  (1248, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492)],
 [(2994, 15.445333480834961),
  (2981, 21.636831283569336),
  (1890, 23.927410125732422),
  (1430, 24.29644012451172),
  (2210, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625)],
 [(325, 16.287860870361328),
  (1496, 22.420625686645508),
  (984, 24.241134643554688),
  (2652, 24.70333480834961),
  (1781, 24.801403045654297),
  (84, 24.87

In [10]:
# os.makedirs("plots/html",exist_ok=True)
# os.makedirs("plots/images",exist_ok=True)   
# # Load the data
# datasets = ["scifact"]
# b_sizes_aug = [10,25,50,100,200]
# k_values=[15]
# max_k = 15
# for data in datasets:
#     make_plots.plot_k_vs_metric(data,max_k,b_sizes_aug)
#     make_plots.plot_b_vs_metric(data,k_values,b_sizes_aug)

In [11]:
with open(f"./pickles/results/greedy_base_0_128_k15_{config['dataset_name']}_bf.pkl", "rb") as f:
    results = pickle.load(f)

In [12]:
results[0]

[(1, 22.11961555480957),
 (3097, 26.389442443847656),
 (2126, 27.00503158569336),
 (1373, 27.201814651489258),
 (2315, 27.21265983581543),
 (3057, 27.217079162597656),
 (0, 27.221275329589844),
 (1248, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492)]

In [13]:
len(results)

323

In [14]:
import copy
configg = copy.deepcopy(config)
configg["mode"] = "disk"
conff = make_config(configg)

['', 'k=15', 'method=v0', 'baseline.bucket_size=0', 'data.dataset_name=nfcorpus', 'embedder.mv_type=colbertv2-plaid', 'embedder.mode=disk']


In [15]:
retriever = get_method(conf)
retriever.run()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3633/3633 [00:00<00:00, 123922.70it/s]



#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: Do Cholesterol Statin Drugs Cause Breast Cancer?, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  2079, 16480,  4244, 27833, 28093,  2378,  5850,  3426,
         7388,  4456,  1029,   102,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

./experiments/nfcorpus/BERT/corpus/compressed_128/batch_0.0.pkl
0 1 loaded
0 2 loaded


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 323/323 [01:06<00:00,  4.89it/s]


[[(1, 22.11961555480957),
  (3097, 26.389442443847656),
  (2126, 27.00503158569336),
  (1373, 27.201814651489258),
  (2315, 27.21265983581543),
  (3057, 27.217079162597656),
  (0, 27.221275329589844),
  (1248, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492),
  (0, 27.222196578979492)],
 [(2994, 15.445333480834961),
  (2981, 21.636831283569336),
  (1890, 23.927410125732422),
  (1430, 24.29644012451172),
  (2210, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625),
  (0, 24.641754150390625)],
 [(325, 16.287860870361328),
  (1496, 22.420625686645508),
  (984, 24.241134643554688),
  (2652, 24.70333480834961),
  (1781, 24.801403045654297),
  (84, 24.87

In [16]:
with open(f"./pickles/results/greedy_base_0_128_k15_{config['dataset_name']}_bf.pkl", "rb") as f:
    results_disk_mode = pickle.load(f)

In [17]:
results_disk_mode[0]

[(1, 22.11961555480957),
 (3097, 26.389442443847656),
 (2126, 27.00503158569336),
 (1373, 27.201814651489258),
 (2315, 27.21265983581543),
 (3057, 27.217079162597656),
 (0, 27.221275329589844),
 (1248, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492),
 (0, 27.222196578979492)]

In [18]:
for i in range(len(results)):
    assert results[i] == results_disk_mode[i], f"Results mismatch at index {i}: {results[i]} != {results_disk_mode[i]}"